<a href="https://colab.research.google.com/github/vivek28n/Medical-RAG-Hallucination-Detection/blob/main/Notebook_04_LLM_Integration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Mon Aug 10 15:56:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import os

print("Python environment ready")
print("Current directory:", os.getcwd())

Python environment ready
Current directory: /content


In [ ]:
%cd /content/Medical-RAG-Hallucination-Detection
!git status


[Errno 2] No such file or directory: '/content/Medical-RAG-Hallucination-Detection'
/content
fatal: not a git repository (or any of the parent directories): .git


In [ ]:
!git clone https://github.com/vivek28n/Medical-RAG-Hallucination-Detection.git

Cloning into 'Medical-RAG-Hallucination-Detection'...
remote: Enumerating objects: 66, done.
remote: Counting objects: 100% (66/66), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 66 (delta 30), reused 28 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (66/66), 1.34 MiB | 3.68 MiB/s, done.
Resolving deltas: 100% (30/30), done.


In [ ]:
%cd /content/Medical-RAG-Hallucination-Detection


/content/Medical-RAG-Hallucination-Detection


In [1]:
!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.1/259.1 kB 25.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.56.3 which is incompatible.


In [2]:
from google import genai

print("Google GenAI imported successfully!")

Google GenAI imported successfully!


In [3]:
from google.colab import userdata

API_KEY = userdata.get("GEMINI_API_KEY")

print("API key loaded:", API_KEY is not None)

API key loaded: True


In [4]:
from google import genai

client = genai.Client(api_key=API_KEY)

print("Gemini client created successfully!")

Gemini client created successfully!


In [6]:
response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents="Say hello and briefly explain what a medical RAG system does."
)

print(response.text)

Hello! 

A medical **RAG (Retrieval-Augmented Generation)** system is a specialized AI tool designed to provide highly accurate, evidence-based answers to clinical or health-related questions.

Here is how it works in a nutshell:

1.  **Retrieval:** Instead of relying solely on the AI's internal training data (which can become outdated), the system searches through a trusted, curated database of medical literature—such as peer-reviewed journals, clinical guidelines, and verified textbooks.
2.  **Augmentation:** It pulls the most relevant documents found during that search and provides them to the AI model as context.
3.  **Generation:** The AI synthesizes that specific, verified information to generate a concise, accurate answer, always citing the sources it used so that clinicians or users can verify the clinical evidence.

Essentially, it acts as a "super-powered librarian" that ensures medical answers are grounded in current, reliable research rather than just general probabilities.

In [7]:
print("chunks available:", "chunks" in globals())
print("FAISS index available:", "index" in globals())
print("Embedding model available:", "embedding_model" in globals())

chunks available: False
FAISS index available: False
Embedding model available: False


In [8]:
import os

print("Current directory:", os.getcwd())
print("Vector DB exists:", os.path.exists("vector_db"))

if os.path.exists("vector_db"):
    print("Vector DB files:", os.listdir("vector_db"))

Current directory: /content
Vector DB exists: False


In [9]:
%cd /content/Medical-RAG-Hallucination-Detection

import os

print("Current directory:", os.getcwd())

[Errno 2] No such file or directory: '/content/Medical-RAG-Hallucination-Detection'
/content
Current directory: /content


In [10]:
import os

print("Folders in /content:")
print(os.listdir("/content"))

Folders in /content:
['.config', 'sample_data']


In [12]:
%cd /content/Medical-RAG-Hallucination-Detection

import os

print("Current directory:", os.getcwd())
print("Notebooks:", os.listdir("notebooks"))

/content/Medical-RAG-Hallucination-Detection
Current directory: /content/Medical-RAG-Hallucination-Detection
Notebooks: ['Notebook_01_Project_Setup.ipynb', 'Notebook_02_Environment_Setup.ipynb', 'Notebook_03_PDF_Text_Extraction.ipynb', '.gitkeep', 'Notebook_04_LLM_Integration.ipynb']


In [13]:
import os

print("Dataset exists:", os.path.exists("dataset"))
print("Raw folder exists:", os.path.exists("dataset/raw"))

if os.path.exists("dataset/raw"):
    print("Raw files:", os.listdir("dataset/raw"))

Dataset exists: True
Raw folder exists: True
Raw files: ['niddk_guiding_principles_diabetes.pdf']


In [14]:
import os

print("Vector DB exists:", os.path.exists("vector_db"))

if os.path.exists("vector_db"):
    print("Vector DB files:", os.listdir("vector_db"))

print("Models exists:", os.path.exists("models"))

if os.path.exists("models"):
    print("Models files:", os.listdir("models"))

Vector DB exists: True
Vector DB files: ['.gitkeep']
Models exists: True
Models files: ['.gitkeep']


In [15]:
import fitz
import re
import faiss
import numpy as np

from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

print("Packages loaded successfully!")

ModuleNotFoundError: No module named 'fitz'

In [16]:
!pip install -q pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 80.1 MB/s eta 0:00:00


In [18]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 77.6 MB/s eta 0:00:00


In [20]:
!pip install -q langchain-text-splitters

In [22]:
pdf_path = "dataset/raw/niddk_guiding_principles_diabetes.pdf"

doc = fitz.open(pdf_path)

print("Number of pages:", len(doc))

Number of pages: 83


In [23]:
text = ""

for page in doc:
    text += page.get_text()

print("Characters extracted:", len(text))
print("\nFirst 1000 characters:\n")
print(text[:1000])

Characters extracted: 201724

First 1000 characters:

1
Guiding Principles
for the Care of People with or at Risk for Diabetes
2
Supporting Organizations
The Guiding Principles for the Care of People with or at Risk for Diabetes was produced by the 
National Diabetes Education Program (NDEP),* a federally funded program sponsored by the 
U.S. Department of Health and Human Services’ National Institutes of Health and Centers for 
Disease Control and Prevention. NDEP’s partnership network includes more than 200 partners 
working together to improve the treatment and outcomes for people with diabetes, promote early 
diagnosis, and prevent or delay the onset of type 2 diabetes. The following organizations support 
the use of the Guiding Principles for the Care of People with or at Risk for Diabetes:
• Academy of Nutrition and Dietetics
• American Academy of Family Physicians
• American Academy of Physician Assistants
• American Association of Clinical Endocrinologists
• American Associatio

In [24]:
pages = []

for page_number, page in enumerate(doc, start=1):
    page_text = page.get_text().strip()

    pages.append({
        "page": page_number,
        "text": page_text
    })

print("Total pages processed:", len(pages))

Total pages processed: 83


In [25]:
def clean_text(text):
    # Multiple spaces ko single space
    text = re.sub(r"[ \t]+", " ", text)

    # Multiple blank lines ko reduce karna
    text = re.sub(r"\n\s*\n+", "\n\n", text)

    # Har line ke extra spaces remove karna
    text = "\n".join(line.strip() for line in text.splitlines())

    return text.strip()


for page in pages:
    page["clean_text"] = clean_text(page["text"])

print("Cleaning completed.")

Cleaning completed.


In [26]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = []

for page in pages:
    page_chunks = splitter.split_text(page["clean_text"])

    for chunk_id, chunk in enumerate(page_chunks):
        chunks.append({
            "chunk_id": f"page_{page['page']}_chunk_{chunk_id}",
            "page": page["page"],
            "text": chunk
        })

print("Total chunks:", len(chunks))

Total chunks: 277


In [27]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [28]:
chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True
)

print("Total embeddings:", len(embeddings))
print("Embedding dimensions:", embeddings.shape[1])

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Total embeddings: 277
Embedding dimensions: 384


In [29]:
embedding_array = np.array(embeddings).astype("float32")

print("Embedding shape:", embedding_array.shape)

dimension = embedding_array.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embedding_array)

print("Total vectors in FAISS:", index.ntotal)

Embedding shape: (277, 384)
Total vectors in FAISS: 277


In [30]:
query = "What are the risk factors for diabetes?"

query_embedding = embedding_model.encode([query])
query_embedding = np.array(query_embedding).astype("float32")

distances, indices = index.search(query_embedding, k=5)

print("Retrieved indices:", indices[0])
print("Distances:", distances[0])

Retrieved indices: [  7  23   8 250   0]
Distances: [0.6567526  0.67704546 0.7259886  0.74482733 0.74978113]


In [31]:
for rank, idx in enumerate(indices[0], start=1):
    print(f"\n--- Retrieved Result {rank} ---")
    print("Chunk ID:", chunks[idx]["chunk_id"])
    print("Page:", chunks[idx]["page"])
    print("Distance:", distances[0][rank - 1])
    print("\nText:")
    print(chunks[idx]["text"][:1000])


--- Retrieved Result 1 ---
Chunk ID: page_5_chunk_0
Page: 5
Distance: 0.6567526

Text:
5
INTRODUCTION
The diabetes problem
Today, 30.3 million people (9.4 percent of the U.S. population) have diabetes, including 7.2 million
who are undiagnosed.1 A major cause of blindness, renal failure, and amputation, diabetes
also increases the risk of cardiovascular disease, cancer, and dementia and more than doubles
individual health care costs.2 The total estimated cost of diagnosed diabetes in 2017 was $327
billion, including $237 billion in direct medical costs and $90 billion in reduced productivity.2
Another 84.1 million Americans (33.9 percent of adults) have glucose levels that are higher than
normal but not high enough to be characterized as diabetes.1 Because persons with these glucose
levels are at increased risk of developing type 2 diabetes, this condition is termed prediabetes
by the Centers for Disease Control and Prevention (CDC) and other organizations.
Proper nutrition and physic

In [32]:
retrieved_context = ""

for rank, idx in enumerate(indices[0], start=1):
    retrieved_context += f"""
--- Evidence {rank} ---
Source page: {chunks[idx]["page"]}
Chunk ID: {chunks[idx]["chunk_id"]}

{chunks[idx]["text"]}
"""

print(retrieved_context)


--- Evidence 1 ---
Source page: 5
Chunk ID: page_5_chunk_0

5
INTRODUCTION
The diabetes problem
Today, 30.3 million people (9.4 percent of the U.S. population) have diabetes, including 7.2 million
who are undiagnosed.1 A major cause of blindness, renal failure, and amputation, diabetes
also increases the risk of cardiovascular disease, cancer, and dementia and more than doubles
individual health care costs.2 The total estimated cost of diagnosed diabetes in 2017 was $327
billion, including $237 billion in direct medical costs and $90 billion in reduced productivity.2
Another 84.1 million Americans (33.9 percent of adults) have glucose levels that are higher than
normal but not high enough to be characterized as diabetes.1 Because persons with these glucose
levels are at increased risk of developing type 2 diabetes, this condition is termed prediabetes
by the Centers for Disease Control and Prevention (CDC) and other organizations.
Proper nutrition and physical activity are the corners

In [33]:
prompt = f"""
You are a medical information assistant.

Answer the user's question using ONLY the provided evidence.
Do not add information that is not supported by the evidence.
If the evidence is insufficient, clearly say that the evidence is insufficient.

User question:
{query}

Retrieved medical evidence:
{retrieved_context}

Instructions:
- Give a concise answer.
- Mention the relevant source page numbers.
- Do not invent medical facts.
"""

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=prompt
)

print(response.text)

Based on the provided evidence, the risk factors for type 2 diabetes include:

*   **Age:** Risk increases with age (page 9).
*   **Weight:** Being overweight or obese (defined as a BMI ≥ 25 kg/m², or ≥ 23 kg/m² for Asian Americans) (page 9).
*   **Family History:** Having a parent or sibling with diabetes (page 9).
*   **High-Risk Populations:** Being African American, Hispanic/Latino, American Indian, Alaska Native, Asian American, or Pacific Islander American (page 9).
*   **Medical History:** Having a history of gestational diabetes mellitus (GDM) (page 9).
*   **Lifestyle Factors:** Physical inactivity (page 9).
*   **Health Conditions:** Hypertension (page 9).
*   **Sleep Issues:** Obstructive sleep apnea and chronic sleep deprivation (less than 6 hours/day) (page 9).
*   **Prediabetes:** Having glucose levels higher than normal but not high enough to be characterized as diabetes (page 5).


In [34]:
import os

notebook_path = "notebooks/Notebook_04_LLM_Integration.ipynb"

print("Notebook exists:", os.path.exists(notebook_path))
print("Notebook path:", notebook_path)

Notebook exists: True
Notebook path: notebooks/Notebook_04_LLM_Integration.ipynb


In [35]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [36]:
!git log -1 --oneline

969c734 (HEAD -> main, origin/main, origin/HEAD) Move Notebook 4 into notebooks directory


In [39]:
!find /content -name "Notebook_04_LLM_Integration.ipynb"

/content/Medical-RAG-Hallucination-Detection/notebooks/Notebook_04_LLM_Integration.ipynb


In [40]:
!git add notebooks/Notebook_04_LLM_Integration.ipynb
!git commit -m "Update Notebook 4 LLM Integration"
!git push origin main

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@66c035c4a6c3.(none)')
fatal: could not read Username for 'https://github.com': No such device or address


In [41]:
!git config --global user.name "Mansi Sharma"
!git config --global user.email "mansisharmapt6@gmail.com"

In [42]:
!git config --global user.name
!git config --global user.email

Mansi Sharma
mansisharmapt6@gmail.com


In [43]:
%cd /content/Medical-RAG-Hallucination-Detection

!git add notebooks/Notebook_04_LLM_Integration.ipynb
!git commit -m "Update Notebook 4 LLM Integration"

/content/Medical-RAG-Hallucination-Detection
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [44]:
import os

path = "/content/Medical-RAG-Hallucination-Detection/notebooks/Notebook_04_LLM_Integration.ipynb"

print("Exists:", os.path.exists(path))
print("Size:", os.path.getsize(path), "bytes")

Exists: True
Size: 6948 bytes


In [45]:
!git diff -- notebooks/Notebook_04_LLM_Integration.ipynb


In [46]:
!find /content -name "Notebook_04_LLM_Integration.ipynb" -type f

/content/Medical-RAG-Hallucination-Detection/notebooks/Notebook_04_LLM_Integration.ipynb


In [47]:
%cd /content/Medical-RAG-Hallucination-Detection

!git status

/content/Medical-RAG-Hallucination-Detection
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [48]:
%cd /content/Medical-RAG-Hallucination-Detection

!git status
!git diff --stat
!git ls-files -v notebooks/Notebook_04_LLM_Integration.ipynb

/content/Medical-RAG-Hallucination-Detection
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
H notebooks/Notebook_04_LLM_Integration.ipynb


In [49]:
import json

path = "/content/Medical-RAG-Hallucination-Detection/notebooks/Notebook_04_LLM_Integration.ipynb"

with open(path, "r", encoding="utf-8") as f:
    nb = json.load(f)

print("Cells:", len(nb["cells"]))

for i, cell in enumerate(nb["cells"]):
    if cell["cell_type"] == "code":
        source = "".join(cell["source"])
        print(f"\n--- Code Cell {i} ---")
        print(source[:300])

Cells: 7

--- Code Cell 1 ---
!nvidia-smi

--- Code Cell 2 ---
import os

print("Python environment ready")
print("Current directory:", os.getcwd())

--- Code Cell 3 ---
import os

print("Python environment ready")
print("Current directory:", os.getcwd())

--- Code Cell 4 ---
%cd /content/Medical-RAG-Hallucination-Detection
!git status


--- Code Cell 5 ---
!git clone https://github.com/vivek28n/Medical-RAG-Hallucination-Detection.git

--- Code Cell 6 ---
%cd /content/Medical-RAG-Hallucination-Detection



In [50]:
import os

print("Current directory:", os.getcwd())
print("Notebook file exists:",
      os.path.exists("/content/Medical-RAG-Hallucination-Detection/notebooks/Notebook_04_LLM_Integration.ipynb"))

Current directory: /content/Medical-RAG-Hallucination-Detection
Notebook file exists: True


In [51]:
import json

path = "/content/Medical-RAG-Hallucination-Detection/notebooks/Notebook_04_LLM_Integration.ipynb"

with open(path, "r", encoding="utf-8") as f:
    nb = json.load(f)

print("Number of cells:", len(nb["cells"]))

for i, cell in enumerate(nb["cells"]):
    print(i, cell["cell_type"], "".join(cell["source"])[:100].replace("\n", " "))


Number of cells: 7
0 markdown <a href="https://colab.research.google.com/github/vivek28n/Medical-RAG-Hallucination-Detection/blob/
1 code !nvidia-smi
2 code import os  print("Python environment ready") print("Current directory:", os.getcwd())
3 code import os  print("Python environment ready") print("Current directory:", os.getcwd())
4 code %cd /content/Medical-RAG-Hallucination-Detection !git status 
5 code !git clone https://github.com/vivek28n/Medical-RAG-Hallucination-Detection.git
6 code %cd /content/Medical-RAG-Hallucination-Detection 


In [24]:
# ============================================
# FAISS RETRIEVAL FUNCTION
# ============================================

def retrieve_documents(question, top_k=5):
    """
    Retrieve the most relevant medical chunks
    from the FAISS index.
    """

    query_embedding = embedding_model.encode(
        [question]
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, idx in enumerate(indices[0], start=1):

        results.append({
            "rank": rank,
            "chunk_id": chunks[idx]["chunk_id"],
            "page": chunks[idx]["page"],
            "text": chunks[idx]["text"],
            "distance": float(distances[0][rank - 1])
        })

    return results


print("✅ Retrieval function created")

✅ Retrieval function created


In [26]:
# ============================================
# BUILD MEDICAL CONTEXT
# ============================================

def build_context(retrieved_docs):

    context_parts = []

    for doc in retrieved_docs:

        context_parts.append(
            f"""[Source Page {doc['page']}]
{doc['text']}"""
        )

    return "\n\n".join(context_parts)


context = build_context(retrieved_docs)

print("✅ Medical context created")
print(context)

✅ Medical context created
[Source Page 15]
AT RISK FOR DIABETES | PRINCIPLE 2

[Source Page 5]
5
INTRODUCTION
The diabetes problem
Today, 30.3 million people (9.4 percent of the U.S. population) have diabetes, including 7.2 million
who are undiagnosed.1 A major cause of blindness, renal failure, and amputation, diabetes
also increases the risk of cardiovascular disease, cancer, and dementia and more than doubles
individual health care costs.2 The total estimated cost of diagnosed diabetes in 2017 was $327
billion, including $237 billion in direct medical costs and $90 billion in reduced productivity.2
Another 84.1 million Americans (33.9 percent of adults) have glucose levels that are higher than
normal but not high enough to be characterized as diabetes.1 Because persons with these glucose
levels are at increased risk of developing type 2 diabetes, this condition is termed prediabetes
by the Centers for Disease Control and Prevention (CDC) and other organizations.
Proper nutrition and

In [28]:
# ============================================
# GROUNDED MEDICAL PROMPT
# ============================================

def create_grounded_prompt(question, context):

    prompt = f"""
You are a medical information assistant operating
inside a Retrieval-Augmented Generation (RAG) system.

Answer the user's question ONLY using the provided
medical evidence.

STRICT RULES:

1. Use only information contained in the evidence.
2. Do not use outside knowledge.
3. Do not invent facts.
4. Do not make unsupported medical claims.
5. If the evidence is insufficient, clearly state:
   "The provided document does not contain sufficient
   evidence to answer this question."
6. Mention relevant source page numbers.
7. Do not provide personalized diagnosis.
8. Do not provide personalized treatment instructions.
9. Keep the answer clear and understandable.

USER QUESTION:
{question}

MEDICAL EVIDENCE:
{context}

ANSWER:
"""

    return prompt


prompt = create_grounded_prompt(
    question,
    context
)

print(prompt)


You are a medical information assistant operating
inside a Retrieval-Augmented Generation (RAG) system.

Answer the user's question ONLY using the provided
medical evidence.

STRICT RULES:

1. Use only information contained in the evidence.
2. Do not use outside knowledge.
3. Do not invent facts.
4. Do not make unsupported medical claims.
5. If the evidence is insufficient, clearly state:
   "The provided document does not contain sufficient
   evidence to answer this question."
6. Mention relevant source page numbers.
7. Do not provide personalized diagnosis.
8. Do not provide personalized treatment instructions.
9. Keep the answer clear and understandable.

USER QUESTION:
What are the risk factors for diabetes?

MEDICAL EVIDENCE:
[Source Page 15]
AT RISK FOR DIABETES | PRINCIPLE 2

[Source Page 5]
5
INTRODUCTION
The diabetes problem
Today, 30.3 million people (9.4 percent of the U.S. population) have diabetes, including 7.2 million
who are undiagnosed.1 A major cause of blindness, r

In [29]:
# ============================================
# GEMINI RESPONSE
# ============================================

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=prompt
)

answer = response.text

print("QUESTION")
print(question)

print("\nANSWER")
print("=" * 70)
print(answer)

QUESTION
What are the risk factors for diabetes?

ANSWER
Based on the provided documents, the following information regarding risk factors for diabetes is available:

*   **Glucose levels:** Individuals with glucose levels that are higher than normal, but not high enough to be classified as diabetes, are at an increased risk of developing type 2 diabetes (Source Page 5).
*   **Obesity/Overweight:** Overweight and obesity in children are identified as risk factors (Source Page 75).

The provided document does not contain a comprehensive list of all risk factors for diabetes.


In [30]:
# ============================================
# COMPLETE MEDICAL RAG PIPELINE
# ============================================

def medical_rag(question, top_k=5):

    # ----------------------------------------
    # 1. Retrieve documents
    # ----------------------------------------

    retrieved_docs = retrieve_documents(
        question,
        top_k=top_k
    )

    # ----------------------------------------
    # 2. Build context
    # ----------------------------------------

    context = build_context(
        retrieved_docs
    )

    # ----------------------------------------
    # 3. Create grounded prompt
    # ----------------------------------------

    prompt = create_grounded_prompt(
        question,
        context
    )

    # ----------------------------------------
    # 4. Generate answer
    # ----------------------------------------

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt
    )

    # ----------------------------------------
    # 5. Return structured result
    # ----------------------------------------

    return {
        "question": question,
        "answer": response.text,
        "retrieved_documents": retrieved_docs
    }


print("✅ Complete Medical RAG pipeline created")

✅ Complete Medical RAG pipeline created


In [31]:
# ============================================
# COMPLETE RAG TEST
# ============================================

result = medical_rag(
    "What are the risk factors for diabetes?"
)

print("QUESTION:")
print(result["question"])

print("\nANSWER:")
print("=" * 70)
print(result["answer"])

print("\nSOURCE PAGES:")

pages_used = sorted(
    set(
        doc["page"]
        for doc in result["retrieved_documents"]
    )
)

print(pages_used)

QUESTION:
What are the risk factors for diabetes?

ANSWER:
Based on the provided documents, the following information regarding risk factors for diabetes is available:

*   **Glucose levels:** Individuals with glucose levels that are higher than normal but not high enough to be classified as diabetes—a condition termed prediabetes—are at an increased risk of developing type 2 diabetes (Source Page 5).
*   **Obesity/Overweight:** Overweight and obesity in children are identified as risk factors associated with diabetes (Source Page 75).

The provided documents do not contain sufficient evidence to provide a comprehensive list of all risk factors for diabetes.

SOURCE PAGES:
[1, 5, 15, 75]


In [32]:
# ============================================
# TEST INSUFFICIENT EVIDENCE
# ============================================

test_question = "What is the capital of France?"

result_test = medical_rag(
    test_question
)

print("QUESTION:")
print(result_test["question"])

print("\nANSWER:")
print("=" * 70)
print(result_test["answer"])

print("\nRETRIEVED PAGES:")

print(
    sorted(
        set(
            doc["page"]
            for doc in result_test["retrieved_documents"]
        )
    )
)

QUESTION:
What is the capital of France?

ANSWER:
The provided document does not contain sufficient evidence to answer this question.

RETRIEVED PAGES:
[11, 25, 42, 52, 56]
